# LeetCode 3251 — Find the Count of Monotonic Pairs II bằng Python

- Link: https://leetcode.com/problems/find-the-count-of-monotonic-pairs-ii/
- Độ khó: **Hard**
- Chủ đề: **DP theo giá trị `arr1[i]`** với ràng buộc kép quy về **một bất đẳng thức** `u ≤ v − max(0, nums[i] − nums[i−1])`, tối ưu bằng **tổng tiền tố** `O(n · M)`; công thức đóng **tổ hợp chia kẹo**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countOfPairs(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Chỉ cần chọn `arr1` (khi đó `arr2[i] = nums[i] − arr1[i]` cố định) với `0 ≤ arr1[i] ≤ nums[i]`.
2. Hai ràng buộc đơn điệu: `arr1[i] ≥ arr1[i−1]` và `arr2[i] ≤ arr2[i−1]` ⇔ `arr1[i] − arr1[i−1] ≥ nums[i] − nums[i−1]`. Gộp: `arr1[i] − arr1[i−1] ≥ d_i := max(0, nums[i] − nums[i−1])`.
3. DP `dp[i][v]` = số cách với `arr1[i] = v`: `dp[i][v] = Σ_{u ≤ v − d_i} dp[i−1][u]` ⇒ tổng tiền tố.
4. Công thức đóng: đặt `b_i = arr1[i] − Σ_{t ≤ i} d_t` (không giảm, `0 ≤ b_i ≤ nums[i] − D_i`); vì dãy cận `nums[i] − D_i` không tăng, chỉ cận cuối quyết định ⇒ `C(nums[n−1] − D + n, n)`.
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums` độ dài `n`. Cặp mảng số nguyên **không âm** `(arr1, arr2)` cùng độ dài `n` là **đơn điệu** nếu `arr1` không giảm, `arr2` không tăng, và `arr1[i] + arr2[i] = nums[i]` với mọi `i`. Đếm số cặp đơn điệu, modulo `10^9 + 7`.

**Ví dụ 1:** `[2,3,2]` → `4`. **Ví dụ 2:** `[5,5,5,5]` → `126`.

**Ràng buộc:** `1 ≤ n ≤ 2000`; `1 ≤ nums[i] ≤ 1000` (bản I: `≤ 50`).

In [1]:
from typing import List
from itertools import product
from math import comb
MOD = 10 ** 9 + 7

examples = [([2, 3, 2], 4), ([5, 5, 5, 5], 126)]
for nums, exp in examples:
    print(nums, '-> kỳ vọng', exp)

[2, 3, 2] -> kỳ vọng 4
[5, 5, 5, 5] -> kỳ vọng 126


## 3. Kiến thức nền

### 3.1. Gộp hai ràng buộc

`arr2[i] ≤ arr2[i−1]` ⇔ `nums[i] − arr1[i] ≤ nums[i−1] − arr1[i−1]` ⇔ `arr1[i] − arr1[i−1] ≥ nums[i] − nums[i−1]`. Cùng với `arr1[i] − arr1[i−1] ≥ 0` ⇒ `arr1[i] − arr1[i−1] ≥ d_i = max(0, nums[i] − nums[i−1])`.

In [2]:
nums = [2, 3, 2]
d = [0] + [max(0, nums[i] - nums[i - 1]) for i in range(1, len(nums))]
print('d =', d, '-> arr1 phải tăng ít nhất d[i] ở mỗi bước')

d = [0, 1, 0] -> arr1 phải tăng ít nhất d[i] ở mỗi bước


### 3.2. Tổng tiền tố cho chuyển trạng thái

`dp[i][v] = Σ_{u=0}^{v − d_i} dp[i−1][u]` với `0 ≤ v ≤ nums[i]` (và `u ≤ nums[i−1]` tự động vì `dp[i−1][u] = 0` ngoài miền) ⇒ `dp[i][v] = P[v − d_i]` với `P` là tiền tố của `dp[i−1]`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `[2, 3, 2]`, `d = [0, 1, 0]`:

- `i = 0`: `arr1[0] ∈ {0, 1, 2}`: `dp = [1, 1, 1]`.
- `i = 1` (`d = 1`, `v ≤ 3`): `dp[v] = Σ_{u ≤ v−1}` ⇒ `[0, 1, 2, 3]`.
- `i = 2` (`d = 0`, `v ≤ 2`): `dp[v] = Σ_{u ≤ v}` ⇒ `[0, 1, 3]`.

Tổng `4` ✓.

## 5. Cách 1 — Brute Force: thử mọi `arr1`

In [3]:
def brute(nums):
    total = 0
    for arr1 in product(*[range(x + 1) for x in nums]):
        arr2 = [x - a for x, a in zip(nums, arr1)]
        if all(arr1[i] <= arr1[i + 1] for i in range(len(nums) - 1)) and \
           all(arr2[i] >= arr2[i + 1] for i in range(len(nums) - 1)):
            total += 1
    return total % MOD

for nums, exp in examples:
    print(brute(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
126 | kỳ vọng 126


**Độ phức tạp:** `O(∏ (nums[i] + 1) · n)`.

## 6. Ý tưởng thuật toán / bước đột phá

```
dp = [1] * (nums[0] + 1)
for i ≥ 1:
    d = max(0, nums[i] − nums[i−1]); P = tiền tố của dp
    new[v] = P[v − d] (nếu v − d ≥ 0, cắt ở độ dài dp), v = 0..nums[i]
answer = Σ dp
```

`O(n · max(nums))` = `2·10^6`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP trực tiếp `O(n · M²)` (đủ cho bản I)

In [4]:
def dp_quadratic(nums):
    dp = [1] * (nums[0] + 1)
    for i in range(1, len(nums)):
        new = [0] * (nums[i] + 1)
        for v in range(nums[i] + 1):
            for u in range(len(dp)):
                if u <= v and nums[i] - v <= nums[i - 1] - u:
                    new[v] = (new[v] + dp[u]) % MOD
        dp = new
    return sum(dp) % MOD

for nums, exp in examples:
    print(dp_quadratic(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
126 | kỳ vọng 126


### 7.2. Cách 3 — DP + tổng tiền tố (lời giải chính)

In [5]:
def dp_prefix(nums):
    dp = [1] * (nums[0] + 1)
    for i in range(1, len(nums)):
        d = max(0, nums[i] - nums[i - 1])
        P = [0] * (len(dp) + 1)
        for u, x in enumerate(dp):
            P[u + 1] = (P[u] + x) % MOD
        new = [0] * (nums[i] + 1)
        for v in range(d, nums[i] + 1):
            new[v] = P[min(v - d + 1, len(dp))]
        dp = new
    return sum(dp) % MOD

for nums, exp in examples:
    print(dp_prefix(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
126 | kỳ vọng 126


`P[min(v − d + 1, len(dp))]` = tổng `dp[0..v−d]` (cắt nếu `v − d` vượt miền của `dp`).

### 7.3. Cách 4 — Công thức chia kẹo

Đặt `D_i = Σ_{t=1}^{i} d_t`, `b_i = arr1[i] − D_i`. Ràng buộc trở thành `b` **không giảm**, `b_0 ≥ 0`, và `b_i ≤ nums[i] − D_i`. Dãy cận `c_i = nums[i] − D_i` **không tăng** (chứng minh ngay dưới) ⇒ ràng buộc chặt nhất là ở `i = n − 1`: `b_{n−1} ≤ L := nums[n−1] − D_{n−1}`. Số dãy không giảm độ dài `n` trong `[0, L]` là `C(L + n, n)`; nếu `L < 0` thì 0.

In [6]:
def stars_and_bars(nums):
    D = sum(max(0, nums[i] - nums[i - 1]) for i in range(1, len(nums)))
    L = nums[-1] - D
    if L < 0:
        return 0
    n = len(nums)
    return comb(L + n, n) % MOD

for nums, exp in examples:
    print(stars_and_bars(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
126 | kỳ vọng 126


Tại sao cận chặt nhất ở cuối? `c_i := nums[i] − D_i` thỏa `c_i − c_{i−1} = (nums[i] − nums[i−1]) − max(0, nums[i] − nums[i−1]) = min(0, nums[i] − nums[i−1]) ≤ 0` ⇒ `c` **không tăng**. Với `b` không giảm, `b_i ≤ b_{n−1} ≤ c_{n−1} ≤ c_i` ⇒ chỉ cần ràng buộc cuối.

## 8. TRACE từng bước

Trace cách 3 trên `[3, 5, 4, 6]`.

In [7]:
def trace(nums):
    dp = [1] * (nums[0] + 1)
    print(f'i = 0: dp = {dp}')
    for i in range(1, len(nums)):
        d = max(0, nums[i] - nums[i - 1])
        P = [0] * (len(dp) + 1)
        for u, x in enumerate(dp):
            P[u + 1] = P[u] + x
        new = [0] * (nums[i] + 1)
        for v in range(d, nums[i] + 1):
            new[v] = P[min(v - d + 1, len(dp))]
        dp = new
        print(f'i = {i} (d = {d}): dp = {dp}')
    D = sum(max(0, nums[i] - nums[i - 1]) for i in range(1, len(nums)))
    print('tổng:', sum(dp), '| công thức C(L + n, n) với L =', nums[-1] - D, '->', stars_and_bars(nums), '| vét cạn:', brute(nums))

trace([3, 5, 4, 6])

i = 0: dp = [1, 1, 1, 1]
i = 1 (d = 2): dp = [0, 0, 1, 2, 3, 4]
i = 2 (d = 0): dp = [0, 0, 1, 3, 6]
i = 3 (d = 2): dp = [0, 0, 0, 0, 1, 4, 10]
tổng: 15 | công thức C(L + n, n) với L = 2 -> 15 | vét cạn: 15


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def countOfPairs(self, nums: List[int]) -> int:
        MOD = 10 ** 9 + 7
        dp = [1] * (nums[0] + 1)
        for i in range(1, len(nums)):
            d = nums[i] - nums[i - 1]
            if d < 0:
                d = 0
            m = len(dp)
            P = [0] * (m + 1)
            acc = 0
            for u in range(m):
                acc += dp[u]
                P[u + 1] = acc % MOD
            new = [0] * (nums[i] + 1)
            for v in range(d, nums[i] + 1):
                t = v - d + 1
                new[v] = P[t if t <= m else m]
            dp = new
        return sum(dp) % MOD

for nums, exp in examples:
    got = Solution().countOfPairs(nums)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
126 OK


## 10. Vì sao thuật toán đúng

1. **Song ánh:** cặp đơn điệu ⇔ `arr1` với `0 ≤ arr1[i] ≤ nums[i]` (để `arr2[i] ≥ 0`) thỏa `arr1[i] − arr1[i−1] ≥ d_i` (mục 3.1).
2. **DP:** `dp[i][v]` = số `arr1[0..i]` hợp lệ có `arr1[i] = v`; chuyển từ mọi `u` với `u ≤ v − d_i` (và `u ≤ nums[i−1]` — tự động bởi miền của `dp[i−1]`).
3. **Tổng tiền tố** chỉ là cách cộng nhanh các `dp[i−1][u]` với `u` trong một đoạn đầu.
4. **Công thức:** mục 7.3 — đếm dãy không giảm độ dài `n` trong `[0, L]` = chọn đa tập `n` phần tử từ `L + 1` giá trị = `C(L + n, n)`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(∏ (nums[i]+1) · n)` | `O(n)` |
| 2. DP trực tiếp | `O(n · M²)` | `O(M)` |
| 3. DP + tiền tố | `O(n · M)` | `O(M)` |
| 4. Công thức | `O(n + L)` (tính tổ hợp) | `O(1)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'dp_quadratic': dp_quadratic, 'dp_prefix': dp_prefix, 'stars_and_bars': stars_and_bars,
         'submit': lambda a: Solution().countOfPairs(a)}
edge = [([1], 2), ([1, 1], 3), ([1, 5], 3), ([3, 1], 3), ([2, 4, 2], 1), ([4, 2, 4], 10)]
for nums, exp in examples + edge:
    assert brute(nums) == exp, (nums, brute(nums))
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


`[1, 5]`: `arr1` phải tăng ít nhất 4 ⇒ `(0, 4), (0, 5), (1, 5)` — 3 cặp; công thức `L = 5 − 4 = 1`, `C(3, 2) = 3` ✓.

### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [10]:
import random
rng = random.Random(3251)
cnt = 0
for _ in range(400):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 5))]
    exp = brute(nums)
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'mảng ngẫu nhiên.')

Khớp vét cạn trên 400 mảng ngẫu nhiên.


### 12.3. Ca lớn `n = 2000`, `nums[i] ≤ 1000`

In [11]:
import time
big = {
    'ngẫu nhiên': [rng.randint(1, 1000) for _ in range(2000)],
    'ngẫu nhiên không tăng': sorted((rng.randint(1, 1000) for _ in range(2000)), reverse=True),
    'dao động nhẹ': [800 + (i % 3) for i in range(2000)],
    'toàn 1000': [1000] * 2000,
    'giảm dần': list(range(1000, 0, -1)) * 2,
}
for name, nums in big.items():
    t = time.time(); a = Solution().countOfPairs(nums); t1 = time.time() - t
    b = stars_and_bars(nums)
    assert a == b
    print(f"{name:12s}: {a:10d} | nộp {t1:.2f}s | khớp công thức")
assert Solution().countOfPairs([1000] * 2000) == comb(1000 + 2000, 2000) % MOD
print('toàn 1000: C(3000, 2000) mod p — đúng.')

ngẫu nhiên  :          0 | nộp 0.09s | khớp công thức


ngẫu nhiên không tăng:       2001 | nộp 0.14s | khớp công thức


dao động nhẹ:          0 | nộp 0.20s | khớp công thức


toàn 1000   :  459897557 | nộp 0.28s | khớp công thức
giảm dần    :          0 | nộp 0.12s | khớp công thức


toàn 1000: C(3000, 2000) mod p — đúng.


Nhận xét: với mảng **ngẫu nhiên** đều trong `[1, 1000]`, tổng các bước tăng `D = Σ max(0, nums[i] − nums[i−1])` cỡ `n · 333` — lớn hơn hẳn `nums[n−1]`, nên `L < 0` và đáp án là `0`. Đó là lý do ta thêm các ca "không tăng" (`D = 0`, `L = nums[n−1]`) và "dao động nhẹ" (`D` cỡ `n · 2/3`, có thể vượt `nums[n−1]`) để kiểm tra những đáp án khác 0. Công thức chia kẹo cho phép **dự đoán trước** đáp án có bằng 0 hay không chỉ bằng một phép cộng.

In [12]:
for name, nums in big.items():
    D = sum(max(0, nums[i] - nums[i - 1]) for i in range(1, len(nums)))
    print(f"{name:22s}: D = {D:7d}, nums[-1] = {nums[-1]:5d}, L = {nums[-1] - D:7d} -> {'0 cặp' if nums[-1] < D else 'có cặp'}")

ngẫu nhiên            : D =  336679, nums[-1] =   626, L = -336053 -> 0 cặp
ngẫu nhiên không tăng : D =       0, nums[-1] =     1, L =       1 -> có cặp
dao động nhẹ          : D =    1333, nums[-1] =   801, L =    -532 -> 0 cặp
toàn 1000             : D =       0, nums[-1] =  1000, L =    1000 -> có cặp
giảm dần              : D =     999, nums[-1] =     1, L =    -998 -> 0 cặp


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ ép `arr1` không giảm (quên `arr2` không tăng)

In [13]:
def wrong_only_arr1(nums):
    dp = [1] * (nums[0] + 1)
    for i in range(1, len(nums)):
        P = [0]
        for x in dp:
            P.append(P[-1] + x)
        dp = [P[min(v + 1, len(dp))] for v in range(nums[i] + 1)]      # SAI: d = 0 luôn
    return sum(dp) % MOD

print('sai :', wrong_only_arr1(examples[0][0]), '| đúng:', Solution().countOfPairs(examples[0][0]))

sai : 10 | đúng: 4


**Vì sao sai:** ví dụ `arr1 = (0, 0, 0)` cho `arr2 = (2, 3, 2)` — không phải không tăng.

### Lỗi 2 — Dùng `d_i = nums[i] − nums[i−1]` (có thể âm) thay vì `max(0, ·)`

In [14]:
def wrong_signed_d(nums):
    dp = [1] * (nums[0] + 1)
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]                                      # SAI khi d < 0
        P = [0]
        for x in dp:
            P.append(P[-1] + x)
        new = [0] * (nums[i] + 1)
        for v in range(nums[i] + 1):
            t = v - d + 1
            if t > 0:
                new[v] = P[min(t, len(dp))]
        dp = new
    return sum(dp) % MOD

print('sai :', wrong_signed_d([3, 1]), '| đúng:', Solution().countOfPairs([3, 1]))

sai : 7 | đúng: 3


**Vì sao sai:** khi `nums` giảm, ràng buộc `arr2` lỏng hơn, nhưng `arr1` **vẫn** phải không giảm (`≥ 0` bước). `d` âm cho phép `arr1` giảm.

### Lỗi 3 — Công thức tổ hợp dùng `nums[0]` (hoặc `min(nums)`) làm cận

In [15]:
def wrong_formula(nums):
    D = sum(max(0, nums[i] - nums[i - 1]) for i in range(1, len(nums)))
    L = min(nums) - D                                                  # SAI: cận đúng là nums[-1] - D
    return comb(L + len(nums), len(nums)) % MOD if L >= 0 else 0

print('sai :', wrong_formula([4, 2, 4]), '| đúng:', Solution().countOfPairs([4, 2, 4]))

sai : 1 | đúng: 10


**Vì sao sai:** dãy cận `c_i = nums[i] − D_i` không tăng nên cận chặt là phần tử **cuối** `c_{n−1}`, không phải `min(nums)` (cần trừ thêm `D`, và `D` tích lũy tới cuối).

### Lỗi 4 — Quên lấy modulo trong tổng tiền tố (tràn số ở ngôn ngữ khác; trong Python thì chậm)

In [16]:
def no_mod_prefix(nums):
    dp = [1] * (nums[0] + 1)
    for i in range(1, len(nums)):
        d = max(0, nums[i] - nums[i - 1])
        P = [0]
        for x in dp:
            P.append(P[-1] + x)                                        # không modulo: số nguyên khổng lồ
        dp = [P[min(v - d + 1, len(dp))] if v >= d else 0 for v in range(nums[i] + 1)]
    return sum(dp)

import time
t = time.time(); big_val = no_mod_prefix([1000] * 400); t1 = time.time() - t
t = time.time(); good = Solution().countOfPairs([1000] * 400); t2 = time.time() - t
print(f'không modulo: {len(str(big_val))} chữ số, {t1:.2f}s | có modulo: {t2:.2f}s | đúng sau khi mod: {big_val % MOD == good}')

không modulo: 363 chữ số, 0.07s | có modulo: 0.07s | đúng sau khi mod: True


**Vì sao sai:** trong C++/Java, tổng tràn `int64` rất nhanh ⇒ kết quả sai; trong Python vẫn đúng nhưng số nguyên dài hàng nghìn chữ số làm chậm hẳn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Hai mảng ràng buộc bởi tổng cố định ⇒ chỉ chọn một mảng.
- Nhiều bất đẳng thức giữa các phần tử kề ⇒ gộp thành một "bước tối thiểu" `d_i`.
- "Dãy không giảm trong `[0, L]`" ⇒ chia kẹo `C(L + n, n)`.

## 15. Biến thể của bài

1. **LC 3250 (bản I):** `nums[i] ≤ 50` — DP `O(n · M²)` đủ.
2. Ba mảng `arr1 + arr2 + arr3 = nums` với ràng buộc đơn điệu — DP hai chiều.
3. `arr1` tăng **nghiêm ngặt** ⇒ `d_i = max(1, nums[i] − nums[i−1])`.
4. Đếm với thêm cận trên `arr1[i] ≤ cap[i]` ⇒ DP (công thức không còn đơn giản).

In [17]:
# Biến thể 3: arr1 tăng nghiêm ngặt
def strict_arr1(nums):
    dp = [1] * (nums[0] + 1)
    for i in range(1, len(nums)):
        d = max(1, nums[i] - nums[i - 1])
        P = [0]
        for x in dp:
            P.append((P[-1] + x) % MOD)
        dp = [P[min(v - d + 1, len(dp))] if v >= d else 0 for v in range(nums[i] + 1)]
    return sum(dp) % MOD

def strict_brute(nums):
    return sum(1 for a in product(*[range(x + 1) for x in nums])
               if all(a[i] < a[i + 1] for i in range(len(a) - 1))
               and all(nums[i] - a[i] >= nums[i + 1] - a[i + 1] for i in range(len(a) - 1)))

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 5))]
    assert strict_arr1(nums) == strict_brute(nums)
print('Biến thể arr1 tăng nghiêm ngặt khớp vét cạn.')

Biến thể arr1 tăng nghiêm ngặt khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Vét cạn | mũ | Oracle |
| 2 | DP trực tiếp | `O(n M²)` | Bản I |
| 3 | DP + tiền tố | `O(n M)` | **Nộp bài** |
| 4 | Công thức chia kẹo | `O(n)` | Đẹp nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [3]`.

**Bài 2 (tính tay):** `nums = [2, 2]`.

**Bài 3 (điền TODO):** liệt kê mọi cặp đơn điệu khi tổng số `≤ 100`.

**Bài 4 (tự viết lại):** biến thể 4 — thêm cận `arr1[i] ≤ cap[i]`.

**Bài 5 (mini challenge):** chứng minh bằng code rằng `c_i = nums[i] − D_i` không tăng trên nhiều mảng ngẫu nhiên, và cho ví dụ mà `L < 0`.

In [18]:
def list_pairs_todo(nums):
    # TODO
    raise NotImplementedError

def with_cap_todo(nums, cap):
    # TODO
    raise NotImplementedError

def check_c_nonincreasing_todo(trials):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `arr1[0] ∈ [0, 3]`. (2) Không ràng buộc kề. (3) `4`.

**Bài 2.** (1) `d = 0`. (2) Dãy không giảm độ dài 2 trong `[0, 2]`. (3) `C(4, 2) = 6`.

**Bài 3.** (1) Quay lui theo `i` với `arr1[i] ≥ arr1[i−1] + d_i`. (2) Ghi lại `arr2`. (3) Số lượng phải bằng đáp án.

**Bài 4.** (1) DP như cách 3 nhưng cắt `v ≤ min(nums[i], cap[i])`. (2) Tiền tố như cũ. (3) Vét cạn.

**Bài 5.** (1) `c_i − c_{i−1} = min(0, nums[i] − nums[i−1])`. (2) Kiểm bằng số. (3) Ví dụ `L < 0`: `[1, 3, 1, 3]` ⇒ `D = 4`, `L = 3 − 4 = −1`.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().countOfPairs([3]) == 4
assert Solution().countOfPairs([2, 2]) == 6

def list_pairs(nums):
    out = []
    def go(i, prev, acc):
        if i == len(nums):
            out.append((tuple(acc), tuple(x - a for x, a in zip(nums, acc)))); return
        d = 0 if i == 0 else max(0, nums[i] - nums[i - 1])
        lo = 0 if i == 0 else prev + d
        for v in range(lo, nums[i] + 1):
            go(i + 1, v, acc + [v])
    go(0, 0, [])
    return out

pairs = list_pairs([2, 3, 2])
assert len(pairs) == 4
print('Bài 1: 4 | Bài 2: 6 | Bài 3:', pairs)

Bài 1: 4 | Bài 2: 6 | Bài 3: [((0, 1, 1), (2, 2, 1)), ((0, 1, 2), (2, 2, 0)), ((0, 2, 2), (2, 1, 0)), ((1, 2, 2), (1, 1, 0))]


In [20]:
# Bài 4, 5
def with_cap(nums, cap):
    top = min(nums[0], cap[0])
    dp = [1] * (top + 1)
    for i in range(1, len(nums)):
        d = max(0, nums[i] - nums[i - 1])
        P = [0]
        for x in dp:
            P.append((P[-1] + x) % MOD)
        top = min(nums[i], cap[i])
        dp = [P[min(v - d + 1, len(dp))] if v >= d else 0 for v in range(top + 1)] if top >= 0 else []
        if not dp:
            return 0
    return sum(dp) % MOD

def check_c_nonincreasing(trials):
    r = random.Random(5)
    for _ in range(trials):
        nums = [r.randint(1, 50) for _ in range(r.randint(1, 20))]
        D, c = 0, [nums[0]]
        for i in range(1, len(nums)):
            D += max(0, nums[i] - nums[i - 1]); c.append(nums[i] - D)
        if any(c[i] > c[i - 1] for i in range(1, len(c))):
            return False
    return True

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(1, 5) for _ in range(rng.randint(1, 4))]
    cap = [rng.randint(0, 5) for _ in nums]
    exp = sum(1 for a in product(*[range(x + 1) for x in nums])
              if all(a[i] <= cap[i] for i in range(len(a)))
              and all(a[i] <= a[i + 1] and nums[i] - a[i] >= nums[i + 1] - a[i + 1] for i in range(len(a) - 1)))
    assert with_cap(nums, cap) == exp
assert check_c_nonincreasing(2000) and Solution().countOfPairs([1, 3, 1, 3]) == 0 == brute([1, 3, 1, 3])
print('Bài 4 khớp vét cạn; Bài 5: c không tăng trên 2000 mảng, [1,3,1,3] có L = -1 -> 0 cặp.')

Bài 4 khớp vét cạn; Bài 5: c không tăng trên 2000 mảng, [1,3,1,3] có L = -1 -> 0 cặp.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
dp = [1] * (nums[0] + 1)
for i ≥ 1: d = max(0, nums[i] - nums[i-1]); P = prefix(dp)
           new[v] = P[min(v-d+1, len(dp))] for v in d..nums[i]
return sum(dp) % MOD          # hoặc C(nums[-1] - Σd + n, n)
```

**Checklist:**

- [ ] Ràng buộc gộp `arr1[i] − arr1[i−1] ≥ max(0, nums[i] − nums[i−1])`?
- [ ] `0 ≤ arr1[i] ≤ nums[i]`?
- [ ] Cắt chỉ số tiền tố ở `len(dp)`?
- [ ] Modulo trong tiền tố?
- [ ] Công thức dùng cận **cuối** `nums[-1] − D`?